In [2]:
#Loading all the necessary modules for the language model and chat model
from langchain_ollama import ChatOllama, OllamaLLM
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain_classic.chains import LLMChain, SequentialChain

chat = ChatOllama(model="mistral")


Language Model - raw LLM-- takes a string in, returns a string


In [3]:
llm = OllamaLLM(model="mistral", temperature=0.2, max_tokens=512)
response = llm.invoke("When did OpenAI models attack HuggingFace in one sentence, accuracy is of importance")
print(response)

 There was no reported incident where OpenAI models attacked HuggingFace. Both are organizations focused on AI research and development, not engaged in hostile actions against each other.


Chat Model- Layer over language model, built for multi-turn convesation with role-tagged messages; not flat messages.

In [4]:
chat = ChatOllama(model="mistral", temperature=0.2, max_tokens=512)
response = chat.invoke("When did OpenAI models attack HuggingFace in one sentence, accuracy is of importance")
print(response)


content=' There was no reported incident where OpenAI models attacked HuggingFace. Both are organizations focused on AI research and development, not engaged in hostile activities against each other.' additional_kwargs={} response_metadata={'model': 'mistral', 'created_at': '2026-07-28T18:30:51.7067182Z', 'done': True, 'done_reason': 'stop', 'total_duration': 5733378700, 'load_duration': 150363800, 'prompt_eval_count': 21, 'prompt_eval_duration': 214510000, 'eval_count': 34, 'eval_duration': 5347576000, 'logprobs': None, 'model_name': 'mistral', 'model_provider': 'ollama'} id='lc_run--019fa9fe-57c0-7652-8f76-1196da969f3d-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 21, 'output_tokens': 34, 'total_tokens': 55}


Chat message - role-tagged objects; what the above model consumes and produces  


In [6]:
messages = [
    SystemMessage(content="You are a an expert scientist that explains complex topics in simple terms."),
    HumanMessage(content="Explain the theory of relativity in simple terms."),
    AIMessage(content="The theory of relativity, developed by Albert Einstein, is a fundamental concept in physics that describes how space and time are interconnected. It consists of two main parts: special relativity and general relativity. Special relativity states that the laws of physics are the same for all observers, regardless of their relative motion, and it introduces the idea that time and space can stretch or contract depending on how fast you are moving. General relativity expands on this by explaining how gravity is not just a force between masses, but rather a curvature of space-time caused by mass and energy. In simple terms, it means that massive objects like planets and stars bend the fabric of space-time around them, which affects how objects move and how time passes."),
    HumanMessage(content="Can you provide a simple analogy to help me understand it better?"),  
]
response = chat.invoke(messages)
print(response)
type(response)  # This will show the type of the response object

content=" Sure! Here's an analogy to help explain the theory of relativity:\n\nImagine space-time is like a rubber sheet stretched out flat. Now, if you place a heavy ball (like a planet) on this sheet, it will cause the sheet to curve around the ball. This curvature affects how other objects move on the sheet. For example, if you roll a marble near the ball, it will follow a curved path instead of moving in a straight line.\n\nThis is similar to what happens with gravity according to general relativity. Massive objects like planets and stars bend space-time around them, causing other objects to move along curved paths. This bending of space-time is what we perceive as gravity.\n\nIn special relativity, the analogy is a bit different. Imagine you are on a train moving at high speed relative to someone standing still on the platform. To you, everything inside the train appears normal, but to the person on the platform, the train seems to be moving and time inside the train seems to slow

langchain_core.messages.ai.AIMessage

Prompt templates - reusable!

In [9]:
from langchain_core.prompts import ChatPromptTemplate

template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that translates English to French."),
    ("human", "Explain {topic} in simple terms."),
])
prompt = template.format_prompt(topic="the theory of relativity")
print(prompt.to_messages())


[SystemMessage(content='You are a helpful assistant that translates English to French.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Explain the theory of relativity in simple terms.', additional_kwargs={}, response_metadata={})]


Output parsers - model's raw text response is coerced into structured type to use with code directly

In [10]:
#PydanticOutputParser here returns a RAGAnswer- I need to read both in detail
parser = StrOutputParser()
chain = template | chat | parser
result = chain.invoke({"topic": "the theory of relativity"})
print(type(result))

<class 'langchain_core.messages.base.TextAccessor'>


Puting these steps together, we create chains as shown above. 
Documents: standard unit for a piece of retrievable text plus its metadata

In [13]:
doc = Document(
    page_content="The theory of relativity, developed by Albert Einstein, is a fundamental concept in physics that describes how space and time are interconnected. It consists of two main parts: special relativity and general relativity. Special relativity states that the laws of physics are the same for all observers, regardless of their relative motion, and it introduces the idea that time and space can stretch or contract depending on how fast you are moving. General relativity expands on this by explaining how gravity is not just a force between masses, but rather a curvature of space-time caused by mass and energy. In simple terms, it means that massive objects like planets and stars bend the fabric of space-time around them, which affects how objects move and how time passes.",
    metadata={"source": "https://en.wikipedia.org/wiki/Theory_of_relativity"}
)


Agents-chain runs a fixed sequence each time it runs

In [15]:
# from langchain_classic.agents import initialize_agent, Tool, AgentType

# def calculator(expr: str) -> str:
#     return str(eval(expr))

# tools = [Tool(name="Calculator", func=calculator, description="Evaluates math expressions")]

# agent = initialize_agent(tools, chat, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION)
# result = agent.invoke("What is 47 * 12, and then explain the result in words?")
# print(result)

Using Chains

In [22]:
template = """Your job is to come up with a classic dish from 
    the area that the user suggests.
    {location}
"""
prompt_template = PromptTemplate(
    input_variables=["location"],
    template=template
)
# chain 1
location_chain = LLMChain(llm=llm, prompt=prompt_template, output_key="location_dish")

template = """Given the dish {location_dish}, come up with a simple recipe on how to make it at home.

    YOUR RESPONSE:
"""
recipe_template = PromptTemplate(
    input_variables=["location_dish"],
    template=template
)
# chain 2
recipe_chain = LLMChain(llm=llm, prompt=recipe_template, output_key="recipe")

template = """Given the recipe {recipe}, come up with a shopping list of ingredients needed to make it."""
shopping_list_template = PromptTemplate(
    input_variables=["recipe"],
    template=template
)
# chain 3
shopping_list_chain = LLMChain(llm=llm, prompt=shopping_list_template, output_key="shopping_list")

overall_chain = SequentialChain(
    chains=[location_chain, recipe_chain, shopping_list_chain],
    input_variables=["location"],
    output_variables=["location_dish", "recipe", "shopping_list"],
    verbose=True
)

overall_chain.invoke({"location": "Mutare"})

C:\Users\PC\AppData\Local\Temp\ipykernel_43056\3538697014.py:10: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 2.0.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  location_chain = LLMChain(llm=llm, prompt=prompt_template, output_key="location_dish")


{'location': 'Mutare',
 'location_dish': " One classic dish from Mutare, Zimbabwe, is Sadza na Nyama, which is a staple meal in Zimbabwean cuisine. Sadza is a type of cornmeal porridge, similar to polenta, and it's often served with Nyama (meat), usually beef or chicken, cooked in a variety of ways such as grilled, stewed, or fried. The dish is typically accompanied by vegetables like cabbage, carrots, and green beans, and it's often enjoyed with ground nuts or peanut butter sauce (Murondo) for added flavor.\n\nAnother popular dish from Mutare is Vetkoek, which are deep-fried dough balls filled with minced meat, onions, and chili sauce. These are a favorite street food in Zimbabwe.\n\nLastly, Samp and Beans is another traditional dish that originates from the Shona people of Zimbabwe, who are the predominant ethnic group in Mutare. Samp is a coarsely ground maize product, similar to hominy, cooked with beans until tender. It's often seasoned with onions, garlic, and salt, making for a 

In [5]:
from langchain_community.document_loaders import PyPDFLoader, WebBaseLoader

C:\Users\PC\AppData\Local\Temp\ipykernel_31896\133196705.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [6]:
pdf_files = [
    "./pdf_context/coctelectricity.pdf",
    # "./pdf_context/chemASsyll.pdf"
]

# Load all documents into one list
all_docs = []
for file in pdf_files:
    loader = PyPDFLoader(file)
    docs = loader.load()
    all_docs.extend(docs)

In [7]:
print(all_docs[0].page_content[:500])  # Print the first 500 characters of the first document

BUYING ELECTRICITY: 
What’s New From 1 July 2025
ELECTRICITY PRICE RELIEF
From 1 July 2025, Cape Town households 
will experience electricity price relief 
despite Eskom’s annual price increase. 
In most other municipalities, electricity 
will go up by 11,32% due to Eskom’s 
increase.
Cape Town’s price relief compensates in 
part for the increased Services and Wires 
Charge for 2025/26.
Price relief will especially help high 
consumption households! Remember to 
consider the lower unit costs for


In [7]:
loader2 = WebBaseLoader("https://www.capetown.gov.za/Family%20and%20home/Residential-utility-services/Residential-electricity-services/the-cost-of-electricity")
web_docs = loader2.load()

In [8]:
from langchain_text_splitters import CharacterTextSplitter

splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = splitter.split_documents(all_docs)

print(f"Number of chunks created: {len(chunks)}")
print(chunks[1].page_content[:1000])  # Preview first chunk

Number of chunks created: 2
LIFELINE ELECTRICITY: MORE RESIDENTS NOW QUALIFY!
Pensioners and social grant recipients 
now qualify for Lifeline Electricity up to 
R27 000 monthly household income! (up 
from R22 000). 
Indigent households qualify for lifeline 
up to R500 000 property value or R7 500 
monthly income.
This is SA’s widest qualifying criteria!
Lifeline customers using 600 units p/m still pay roughly the same as they did 
three years ago. 
Customers need to stay within the 450-unit monthly average over 12 months to 
remain on the Lifeline tariff. 
Lifeline customers receive either 25 or 60 units Free Basic Electricity if they have 
a prepaid meter installed depending on consumption level. Contact us to get 
your meter!
Special Price Protection continues
CITY-WIDE CLEANING NO LONGER FUNDED VIA ELECTRICITY 
CONTRIBUTION IN RATES
Previously, all City-supplied customers 
contributed to City-Wide Cleaning 
services via the cost of electricity into the 
rates account. 
City-Wide Cl

In [11]:
from langchain_ollama import OllamaEmbeddings

embedding_model = OllamaEmbeddings(model="nomic-embed-text")
texts = [doc.page_content for doc in chunks]
embedding_result = embedding_model.embed_documents(texts)
print(embedding_result[0][:5]) 


[0.037827473, 0.08263213, -0.2260274, -0.036463104, 0.06696225]


In [9]:
from langchain_chroma import Chroma

In [12]:
docsearch = Chroma.from_documents(chunks, embedding_model)
docs = docsearch.similarity_search("Langchain")
print(docs[0].page_content)

BUYING ELECTRICITY: 
What’s New From 1 July 2025
ELECTRICITY PRICE RELIEF
From 1 July 2025, Cape Town households 
will experience electricity price relief 
despite Eskom’s annual price increase. 
In most other municipalities, electricity 
will go up by 11,32% due to Eskom’s 
increase.
Cape Town’s price relief compensates in 
part for the increased Services and Wires 
Charge for 2025/26.
Price relief will especially help high 
consumption households! Remember to 
consider the lower unit costs for electricity 
when calculating your new total monthly 
bill for 2025/26.
FIXED CHARGES: HOW IT WORKS
• Customers now pay a total R59,90 (VAT excl.) fixed   
 charge per month, called the Services and Wires Charge.
• The charge is divided into a daily rate of R1,97. 
• With each electricity purchase, customers pay for the   
 number of days since the last purchase. 
• Example: a customer buying electricity every 15 days will  
 pay 15 x R1,97 = R29,55 in fixed charges (R59,90 for   
 30 days). 
D

In [13]:
retriever = docsearch.as_retriever()
retriever.invoke("Langchain")

[Document(id='aa485ec3-662b-48cf-bca8-3e9666039184', metadata={'page': 0, 'total_pages': 2, 'source': './pdf_context/coctelectricity.pdf', 'producer': 'Adobe PDF Library 17.0', 'creationdate': '2025-08-05T09:02:07+02:00', 'creator': 'Adobe InDesign 20.4 (Macintosh)', 'moddate': '2025-08-05T09:02:07+02:00', 'trapped': '/False', 'page_label': '1'}, page_content='BUYING ELECTRICITY: \nWhat’s New From 1 July 2025\nELECTRICITY PRICE RELIEF\nFrom 1 July 2025, Cape Town households \nwill experience electricity price relief \ndespite Eskom’s annual price increase. \nIn most other municipalities, electricity \nwill go up by 11,32% due to Eskom’s \nincrease.\nCape Town’s price relief compensates in \npart for the increased Services and Wires \nCharge for 2025/26.\nPrice relief will especially help high \nconsumption households! Remember to \nconsider the lower unit costs for electricity \nwhen calculating your new total monthly \nbill for 2025/26.\nFIXED CHARGES: HOW IT WORKS\n• Customers now pa

In [14]:
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.storage import InMemoryStore

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400)

store = InMemoryStore()
vectorstore = Chroma(embedding_function=embedding_model, collection_name="parent_child_demo")

retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)
retriever.add_documents(all_docs)

sub_docs = vectorstore.similarity_search("Langchain")   # small chunk matched
retrieved_docs = retriever.invoke("Langchain")           # full parent returned

In [15]:
print("--- sub_docs (small child chunk) ---")
print(len(sub_docs[0].page_content), "characters")
print(sub_docs[0].page_content)

print("\n--- retrieved_docs (full parent chunk) ---")
print(len(retrieved_docs[0].page_content), "characters")
print(retrieved_docs[0].page_content)


--- sub_docs (small child chunk) ---
368 characters
• Servicing informal settlements. 
• Servicing green litter bins and large   
 central business districts bins/   
 containers. 
• Animal carcass removals. 
• Unscheduled residential cleaning   
 where resources allow. 
While City-supplied customers have 
always contributed to City-Wide 
Cleaning via electricity purchases, this 
is a new charge for Eskom-supply area

--- retrieved_docs (full parent chunk) ---
1990 characters
LIFELINE ELECTRICITY: MORE RESIDENTS NOW QUALIFY!
Pensioners and social grant recipients 
now qualify for Lifeline Electricity up to 
R27 000 monthly household income! (up 
from R22 000). 
Indigent households qualify for lifeline 
up to R500 000 property value or R7 500 
monthly income.
This is SA’s widest qualifying criteria!
Lifeline customers using 600 units p/m still pay roughly the same as they did 
three years ago. 
Customers need to stay within the 450-unit monthly average over 12 months to 
remain on the L

Switching to LCEL and leaving legacy code

In [17]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [19]:
def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer only using the provided context."),
    ("human", "Context:\n{context}\n\nQuestion: {question}"),
])

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt | chat | StrOutputParser()
)
rag_chain.invoke("How is electricity charged each time a user goes to buy a token?")

' Each time a user goes to buy a token, they pay for the number of days since the last purchase. The total fixed charge per month, called the Services and Wires Charge, is R59,90 (VAT excluded). This charge is divided into a daily rate of R1,97. So, when buying a token, the user pays for the number of days elapsed since their last purchase multiplied by R1,97. For example, if a customer buys electricity every 15 days, they will pay 15 x R1,97 = R29,55 in fixed charges (R59,90 for 30 days). Additionally, the user also pays for the electricity units consumed based on the tariff blocks.'

Chat message history-the raw list of messages

In [20]:
from langchain_community.chat_message_histories import ChatMessageHistory

history = ChatMessageHistory()
history.add_user_message("Hi, who are you?")
ai_response = chat.invoke(history.messages)
history.add_ai_message(ai_response.content)

Conversation buffer memory-auto management of using ConversationChain is now legacy...

In [21]:
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}
def get_session_history(session_id):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

chat_with_memory = RunnableWithMessageHistory(chat, get_session_history)
config = {"configurable": {"session_id": "user-1"}}

chat_with_memory.invoke([HumanMessage(content="Hi, I am Ngonied.")], config=config)
chat_with_memory.invoke([HumanMessage(content="Who am I?")], config=config)

C:\Users\PC\AppData\Roaming\Python\Python314\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


AIMessage(content=' You are Ngonied, as you introduced yourself earlier in the conversation. Is there something specific you would like to know about yourself or need help with?', additional_kwargs={}, response_metadata={'model': 'mistral', 'created_at': '2026-07-28T20:24:36.106812Z', 'done': True, 'done_reason': 'stop', 'total_duration': 7408999400, 'load_duration': 109126800, 'prompt_eval_count': 41, 'prompt_eval_duration': 1712367000, 'eval_count': 32, 'eval_duration': 5570043000, 'logprobs': None, 'model_name': 'mistral', 'model_provider': 'ollama'}, id='lc_run--019faa66-730f-7ef3-93c8-30c2a6bca4df-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 41, 'output_tokens': 32, 'total_tokens': 73})

Tools-giving the model something to do

In [23]:
from langchain_core. tools import tool

@tool
def python_calcualtor(code: str) -> str:
    """Evaluates a Python expression and returns the result."""
    
    return str(eval(code))
        
@tool
def search_weather(location: str) -> str:
    """Searches for the current weather in a given location."""
    
    # Placeholder implementation; in a real scenario, one would call a weather API
    return f"The current weather in {location} is sunny with a temperature of 25°C."

then the toolkit

In [24]:
tools = [python_calcualtor, search_weather]

Agents-model deciding which tool to use thought>Action>observation loop

In [29]:
from langchain_core.prompts import PromptTemplate
from langchain_ollama import ChatOllama
from langchain_classic.agents import create_react_agent, AgentExecutor

# temperature=0 for agent reasoning - format-following benefits from less creative phrasing
agent_chat = ChatOllama(model="mistral", temperature=0)

react_prompt = PromptTemplate.from_template("""Answer the following questions as best you can. You have access to the following tools:

{tools}

IMPORTANT: You must follow the format below exactly. Once you know the final answer, respond with exactly:
Final Answer: <your answer>
Do not paraphrase "Final Answer:" or write it any other way — use that exact text, on its own line.

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}""")

agent = create_react_agent(llm=agent_chat, tools=tools, prompt=react_prompt)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=3,   # caps it here instead of looping forever if it still misformats
)

agent_executor.invoke({"input": "What's 345 * 789?"})



> Entering new AgentExecutor chain...
 I need to use the python_calcualtor tool to evaluate the given expression.
Action: python_calcualtor
Action Input: "345 * 789"272205 Final Answer: 272205

> Finished chain.


{'input': "What's 345 * 789?", 'output': '272205'}